In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
import numpy as np
import numpy.typing as npt


from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp



from qubo_qaoa.utils.lr_qaoa import get_LR_qaoa_circuit

from qiskit_qaoa.utils.hamiltonian_utils import get_Q_and_hamiltonian
from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples

In [2]:
service = QiskitRuntimeService(name='eu_test_instance')
backend = service.least_busy(min_num_qubits=1, operational=True, simulator=False) 

In [3]:
backend

<IBMBackend('ibm_brussels')>

In [24]:
filename = 'test_N2_W2'
N = 2
p = 1
delta_b_fixed = 0.63
delta_g_fixed = 0.16


data_file = f'/lustre/scratch127/qpg/jc59/new_qubo_formulation/oriented/qubo_data/qubo_data_{filename}.gfa.pkl'


Q, hamiltonian, offset, ising_offset = get_Q_and_hamiltonian(data_file)
num_qubits: int = hamiltonian.num_qubits

   
phis = ParameterVector('ϕ', num_qubits)
fixed_qc, circuit = get_LR_qaoa_circuit(
    p, delta_b_fixed, delta_g_fixed, num_qubits,
    hamiltonian, None, phis=phis, measure=True
)
fixed_qc.remove_final_measurements()

09:33:18 - qubo_qaoa.utils.lr_qaoa - INFO - p = 1. Circuit depth: 18


In [25]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

In [26]:
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")


In [27]:
catalog.list()


[QiskitFunction(algorithmiq/tem), QiskitFunction(qedma/qesem)]

In [28]:
qesem_function = catalog.load("qedma/qesem")
if qesem_function is None:
    raise Exception()

In [ ]:
prob = 1 / (2 * N)
theta = 2 * np.arcsin(np.sqrt(prob))
init_angles = theta * np.ones((num_qubits,))
sample_circuit = fixed_qc.assign_parameters(init_angles, inplace=False)

In [30]:
sample_circuit.draw(fold=-1)

┌─────────┐┌───────────┐                                                                                                                     ┌──────────┐┌───────────┐┌─────────┐                                                                                                                                      
q_0: ┤ Ry(π/3) ├┤ Rz(-1.56) ├─■──────────■───────────────────■───────────────────■────────────────────────────────■─────────────────────■─────────┤ Ry(-π/3) ├┤ Rz(-0.63) ├┤ Ry(π/3) ├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     ├─────────┤├───────────┤ │ZZ(0.88)  │                   │                   │                                │                     │         ├──────────┤├───────────┤├─────────┤                                                                                                                                      
q_1: ┤ Ry(π/3) ├┤ Rz(-1.76) ├─■──────────┼─────────■─────────┼─────────■─────────┼─────────────────────■──────────┼──────────■──────────┼─────────┤ Ry(-π/3) ├┤ Rz(-0.63) ├┤ Ry(π/3) ├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     ├─────────┤├───────────┤            │ZZ(0.8)  │ZZ(0.8)  │         │         │                     │          │          │          │         └──────────┘└───────────┘└─────────┘                      ┌──────────┐┌───────────┐┌─────────┐                                                                            
q_2: ┤ Ry(π/3) ├┤ Rz(-1.76) ├────────────■─────────■─────────┼─────────┼─────────┼──────────■──────────┼──────────┼──────────┼──────────┼───────────────────────■─────────────────────────────────■─────────┤ Ry(-π/3) ├┤ Rz(-0.63) ├┤ Ry(π/3) ├────────────────────────────────────────────────────────────────────────────
     ├─────────┤├───────────┤                                │ZZ(0.8)  │ZZ(0.8)  │          │ZZ(0.88)  │          │          │          │                       │                                 │         └──────────┘└───────────┘├─────────┴┐┌───────────┐┌─────────┐                                                   
q_3: ┤ Ry(π/3) ├┤ Rz(-1.56) ├────────────────────────────────■─────────■─────────┼──────────■──────────┼──────────┼──────────┼──────────┼──────────■────────────┼──────────────────────■──────────┼───────────────────────■──────────┤ Ry(-π/3) ├┤ Rz(-0.63) ├┤ Ry(π/3) ├───────────────────────────────────────────────────
     ├─────────┤├───────────┤                                                    │ZZ(0.08)             │ZZ(0.08)  │          │          │          │            │                      │          │                       │          └──────────┘└───────────┘├─────────┴┐┌───────────┐ ┌─────────┐                         
q_4: ┤ Ry(π/3) ├┤ Rz(-1.76) ├────────────────────────────────────────────────────■─────────────────────■──────────┼──────────┼──────────┼──────────┼────────────┼───────────■──────────┼──────────┼───────────■───────────┼─────────────────────────■─────────┤ Ry(-π/3) ├┤ Rz(-0.63) ├─┤ Ry(π/3) ├─────────────────────────
     ├─────────┤├───────────┤                                                                                     │ZZ(0.08)  │ZZ(0.08)  │          │ZZ(-0.2)    │           │ZZ(0.88)  │          │           │           │                         │         └──────────┘└┬──────────┤┌┴─────────┴┐ ┌─────────┐            
q_5: ┤ Ry(π/3) ├┤ Rz(-1.56) ├─────────────────────────────────────────────────────────────────────────────────────■──────────■──────────┼──────────■────────────┼───────────■──────────┼──────────┼───────────┼───────────┼────────────■────────────┼───────────■──────────┤ Ry(-π/3) ├┤ Rz(-0.63) ├─┤ Ry(π/3) ├────────────
     ├─────────┤├───────────┤                                                                                                           │ZZ(-0.2)               │ZZ(0.08)              │ZZ(0.08)  │      

In [32]:
sample_circuit.count_ops()


OrderedDict([('ry', 24), ('rzz', 22), ('rz', 16)])

In [31]:
obsvs = [
    SparsePauliOp.from_sparse_list(
        [('Z', [q], 1)], num_qubits=circuit.num_qubits
    ) for q in range(fixed_qc.num_qubits)
][:1]


time_estimation_job = qesem_function.run(
    pubs=[(sample_circuit, obsvs)],
    options={
        "estimate_time_only": "analytical",
        "transpilation_level": 1,
        "default_precision": 0.1
    },
    backend_name=backend.name,
)
time_estimation_job.result()

PrimitiveResult([PubResult(data=DataBin(), metadata={'time_estimation_sec': 3600, 'description': None, 'total_qpu_time': 0})], metadata={})

In [34]:
print(f"""
      Estimated time to estimate observable in hours:
      Default settings, 1 single-qubit observable: {48600 // 3600}
      transpilation_level=1, default_precision=0.1, 1 single-qubit observable: {3600 // 3600}
      """)


      Estimated time to estimate observable in hours:
      Default settings, 1 single-qubit observable: 13
      transpilation_level=1, default_precision=0.1, 1 single-qubit observable: 1
      


In [ ]:
48600 // 60 # Original settings, single obsv, with mmt
3600 // 60 # New settings, single obsv, with mmt
3600 // 60 # New settings, single obsv, without mmt

810

In [17]:
t_sample_circuit = transpile(sample_circuit, backend, optimization_level=3)

In [18]:
t_sample_circuit.count_ops()

OrderedDict([('rz', 231),
             ('sx', 150),
             ('ecr', 78),
             ('x', 16),
             ('measure', 8),
             ('barrier', 1)])

In [19]:
t_sample_circuit.draw(fold=-1)

global phase: 3π/2
          ┌────┐┌───────────┐┌────┐┌──────────┐                                                                                                                                                                                                                                                                                                                                                                                ┌──────┐                                                        ┌──────┐   ┌─────────────┐                                                                                                                                                                                           ┌──────┐                                          ┌──────┐  ┌────────┐     ┌────┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                  ┌──────┐┌──────────┐┌────┐ ┌────────┐┌──────┐┌─────────┐ ┌────┐┌─────────┐┌──────┐   ┌────┐   ┌──────────┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         ░    ┌─┐                  
 q_4 -> 6 ┤ √X ├┤ Rz(-2π/3) ├┤ √X ├┤ Rz(-π/2) ├────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤0     ├────────────────────────────────────────────────────────┤0     ├───┤ Rz(-0.1892) ├───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤0     ├──────────────────────────────────────────┤0     ├──┤ Rz(-π) ├─────┤ √X ├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤0     ├┤ Rz(-π/2) ├┤ √X ├─┤ Rz(-π) ├┤0     ├┤ Rz(π/2) ├─┤ √X ├┤ Rz(π/2) ├┤0     ├───┤ √X ├───┤ Rz(-π/2) ├───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [35]:
qesem_function = catalog.load("qedma/qesem")
if qesem_function is None:
    raise Exception()

obsvs = [
    SparsePauliOp.from_sparse_list(
        [('Z', [q], 1)], num_qubits=t_sample_circuit.num_qubits
    ) for q in range(t_sample_circuit.num_qubits)
][:1]

transpiled_time_estimation_job = qesem_function.run(
    pubs=[(t_sample_circuit, obsvs)],
    options={
        "estimate_time_only": "analytical",
        "transpilation_level": 1,
        "default_precision": 0.1
    },
    backend_name=backend.name,
)
transpiled_time_estimation_job.result()

PrimitiveResult([PubResult(data=DataBin(), metadata={'time_estimation_sec': 1800, 'description': None, 'total_qpu_time': 0})], metadata={})

In [ ]:
qesem_function = catalog.load("qedma/qesem")
if qesem_function is None:
    raise Exception()

obsvs = [
    SparsePauliOp.from_sparse_list(
        [('Z', [q], 1)], num_qubits=t_sample_circuit.num_qubits
    ) for q in range(t_sample_circuit.num_qubits)
][:1]

transpiled_time_estimation_job = qesem_function.run(
    pubs=[(t_sample_circuit, obsvs)],
    options={
        "estimate_time_only": "empirical",
        "transpilation_level": 1,
        "default_precision": 0.1
    },
    backend_name=backend.name,
)
transpiled_time_estimation_job.result()

In [7]:
job = service.job('d5jrhi31989c73eisrdg')
job_result = job.result()
job_result

In [15]:
# results = time_estimation_job.result()[0]
# print(f"Mitigated expectation values: \n{results.data.evs}")
# print(f"Mitigated error-bar: \n{results.data.stds}")
# noisy_results = results.metadata["noisy_results"]
# print(f"Noisy expectation values: \n{noisy_results.evs}")
# print(f"Noisy error-bar: \n{noisy_results.stds}")
# print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
# print(
#     f"Gates fidelity measured during the experiment: \n {results.metadata['gate_fidelities']}"
# )
# print(
#     f"Total shots / mitigation shots: \n {results.metadata['total_shots']} / {results.metadata['mitigation_shots']}"
# )
# print("Transpiled circuits:")
# for i, circuit in enumerate(results.metadata["transpiled_circs"]):
#     print(f"Circuit {i}:")
#     print(f"  Circuit: \n {circuit['circuit']}")
#     print(f"  Qubit mapping: \n {circuit['qubit_map']}")
#     print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")